# 01 Load reference data

Loads the site's reference data from the **Files** area of the Lakehouse into Delta tables.

The files were uploaded from the edge server by `edge/scripts/07-sync-reference-data.sh`:

| Files folder | Content | Tables created |
|---|---|---|
| `Files/reference/*.csv` | Site, turbines, components with warranty, alarm catalogue, power curve, spare parts, technicians, on-call roster, contracts, SOP index | `ref_site`, `ref_turbines`, `ref_components`, `ref_alarm_codes`, `ref_power_curve`, `ref_spare_parts`, `ref_technicians`, `ref_oncall_roster`, `ref_contracts`, `ref_sop_index` |
| `Files/reference/gis/*.geojson` | Turbine positions, roads, infrastructure, site boundary | `ref_gis_features` |
| `Files/reference/terrain/heightmap.json` | Terrain elevation grid | `ref_terrain_grid` |
| `Files/sop/*.md` | Standard operating procedures | `ref_sop_documents` |

Work orders and everything else that has a date are operating history; notebook `02_generate_history` creates them.

**Before you run it:** attach the Lakehouse that holds these files as the notebook's default Lakehouse (Explorer pane, *Add data items*). The notebook is safe to run again; every table is overwritten.

In [ ]:
# Locations inside the default Lakehouse of this notebook.
REFERENCE_DIR = "Files/reference"            # Spark path
LOCAL_FILES = "/lakehouse/default/Files"     # the same Files area as a local path, for plain Python

## Master data: CSV files to Delta tables

In [ ]:
from pyspark.sql import functions as F

# file name -> (table name, columns to store as dates)
CSV_TABLES = {
    "site.csv": ("ref_site", []),
    "turbines.csv": ("ref_turbines", ["commissionedDate"]),
    "components.csv": ("ref_components", ["installDate", "warrantyEndDate"]),
    "alarm_codes.csv": ("ref_alarm_codes", []),
    "power_curve.csv": ("ref_power_curve", []),
    "spare_parts.csv": ("ref_spare_parts", []),
    "technicians.csv": ("ref_technicians", []),
    "oncall_roster.csv": ("ref_oncall_roster", []),
    "contracts.csv": ("ref_contracts", ["startDate", "endDate"]),
    "sop_index.csv": ("ref_sop_index", []),
}

for file_name, (table, date_columns) in CSV_TABLES.items():
    df = (spark.read
          .option("header", True)
          .option("inferSchema", True)
          .option("multiLine", True)
          .option("quote", '"')
          .option("escape", '"')
          .csv(f"{REFERENCE_DIR}/{file_name}"))
    for column in date_columns:
        df = df.withColumn(column, F.to_date(F.col(column).cast("string")))
    df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(table)
    print(f"{table}: {df.count()} rows")

# Replaced by ops_work_orders (notebook 02) in version 1.1 of the repository.
spark.sql("DROP TABLE IF EXISTS ref_maintenance_history")

## GIS layers: one row per map feature

Geometry is kept as GeoJSON text so that any map visual or application can use it.

In [ ]:
import glob
import json
import os


def read_gis_features(gis_dir):
    rows = []
    for path in sorted(glob.glob(os.path.join(gis_dir, "*.geojson"))):
        layer = os.path.splitext(os.path.basename(path))[0]
        with open(path, encoding="utf-8") as f:
            collection = json.load(f)
        for feature in collection["features"]:
            props = feature["properties"]
            rows.append((
                layer,
                str(props.get("id")),
                props.get("name"),
                props.get("assetType") or props.get("roadType") or layer,
                feature["geometry"]["type"],
                json.dumps(feature["geometry"]),
                json.dumps(props),
            ))
    return rows


gis_rows = read_gis_features(f"{LOCAL_FILES}/reference/gis")
gis_schema = ("layer string, featureId string, name string, featureType string, "
              "geometryType string, geometryGeoJson string, propertiesJson string")
(spark.createDataFrame(gis_rows, gis_schema)
 .write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("ref_gis_features"))
print(f"ref_gis_features: {len(gis_rows)} rows")

## Terrain: elevation grid

One row per grid point, in local site coordinates (metres east and north of the site origin). The 3D app in a later phase reads the original `heightmap.json`; this table makes the same terrain available to SQL and reports.

In [ ]:
def read_terrain_grid(heightmap_path):
    with open(heightmap_path, encoding="utf-8") as f:
        terrain = json.load(f)
    half = terrain["sizeM"] / 2
    spacing = terrain["spacingM"]
    rows = []
    for r, row in enumerate(terrain["elevationsM"]):          # south to north
        for c, elevation in enumerate(row):                    # west to east
            rows.append((r, c, -half + c * spacing, -half + r * spacing, float(elevation)))
    return rows


terrain_rows = read_terrain_grid(f"{LOCAL_FILES}/reference/terrain/heightmap.json")
(spark.createDataFrame(terrain_rows, "rowIndex int, colIndex int, localX double, localY double, elevationM double")
 .write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("ref_terrain_grid"))
print(f"ref_terrain_grid: {len(terrain_rows)} rows")

## SOP documents: full text in a table

The markdown files stay in `Files/sop`. This table holds the same text, keyed by SOP id, so that agents and queries in later phases can look procedures up by the `sopId` carried on every alarm.

In [ ]:
def read_sop_documents(sop_dir):
    rows = []
    for path in sorted(glob.glob(os.path.join(sop_dir, "*.md"))):
        file_name = os.path.basename(path)
        with open(path, encoding="utf-8") as f:
            content = f.read()
        first_line = content.splitlines()[0].lstrip("# ").strip()
        sop_id, _, title = first_line.partition(":")
        rows.append((sop_id.strip(), title.strip(), f"sop/{file_name}", content))
    return rows


sop_rows = read_sop_documents(f"{LOCAL_FILES}/sop")
(spark.createDataFrame(sop_rows, "sopId string, title string, file string, content string")
 .write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("ref_sop_documents"))
print(f"ref_sop_documents: {len(sop_rows)} rows")

## Check

Expected: 12 turbines, 132 components, 40 alarm codes (17 of them raised by the live simulator), 21 spare parts, 11 people, 4 contracts and 7 SOP documents. The last query lists the contracts.

In [ ]:
for table in ["ref_site", "ref_turbines", "ref_components", "ref_alarm_codes", "ref_power_curve",
              "ref_spare_parts", "ref_technicians", "ref_oncall_roster", "ref_contracts", "ref_sop_index",
              "ref_gis_features", "ref_terrain_grid", "ref_sop_documents"]:
    print(f"{table:28s} {spark.table(table).count():6d} rows")

display(spark.sql("""
    SELECT contractId, contractType, counterparty, startDate, endDate, priceUsdPerMwh, guaranteedAvailabilityPct
    FROM ref_contracts
    ORDER BY contractId
"""))